# 作业 01：一个预测函数，接收多个输入张量

上周的预测函数只有一个输入 x。这次同一个样本有两个输入 x1、x2，它们共同决定一个目标 y：

$$\hat y = w_1x_1 + w_2x_2 + b.$$

重点：调用一次预测函数时，需要同时传入 x1 和 x2。它们不是两次独立预测，也不是两个不同样本；例如同一个物体的两个测量值，都可能影响预测结果。

下面还会把多个样本分别放进 x1、x2 和 y。`x1[i]`、`x2[i]`、`y[i]` 属于同一个样本。请区分“一个样本有两个输入”和“共有多少个样本”。

本题仅需 CPU 和 PyTorch。训练步骤沿用上周：预测、损失、清空梯度、反向传播、更新参数。


In [1]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b

# 同一个样本的两个输入，预测时必须一起使用
x1_demo = torch.tensor(1.0)
x2_demo = torch.tensor(2.0)
y_demo = torch.tensor(-3.0)
w1_demo = torch.tensor(1.0)
w2_demo = torch.tensor(1.0)
b_demo = torch.tensor(0.0)
y_pred_demo = predict(x1_demo, x2_demo, w1_demo, w2_demo, b_demo)
loss_demo = torch.abs(y_demo - y_pred_demo)
for name, value in [('x1', x1_demo), ('x2', x2_demo), ('y', y_demo),
                    ('y_pred', y_pred_demo), ('loss', loss_demo)]:
    print(name, 'shape =', tuple(value.shape), 'value =', value)



x1 shape = () value = tensor(1.)
x2 shape = () value = tensor(2.)
y shape = () value = tensor(-3.)
y_pred shape = () value = tensor(3.)
loss shape = () value = tensor(6.)


## 同一个样本，也可以只用一个 x 张量

上面的 x1_demo 和 x2_demo 分别是两个标量张量，shape 都为 `()`。现在直接用 `torch.tensor([1.0, 2.0])` 创建一个张量 x，shape 就是 `(2,)`：**一个样本，包含两个输入值**。这里的 2 表示两个特征，不是两个样本。

预测公式和参数都不变，只改变函数接收输入的方式：原来传入 x1、x2；现在传入 x，再用 x[0]、x[1] 取出两个值。此处只比较单个样本，不进行训练；多样本的训练代码放在作业 02。


In [3]:
# 把原来分开的两个输入 x1=1.0、x2=2.0，打包成一个一维张量
# shape 是 (2,)，表示这个张量里有2个元素
# x[0] 就是原来的 x1，x[1] 就是原来的 x2
x = torch.tensor([1.0, 2.0])

# 预测函数（打包版）：输入是一个包含两个值的张量 x
def predict_packed(x, w1, w2, b):
    # x[0] 取第一个元素（对应原来的 x1），x[1] 取第二个元素（对应原来的 x2）
    # 公式：ŷ = w1*x1 + w2*x2 + b
    return w1 * x[0] + w2 * x[1] + b

# 用打包后的方式算预测值
y_pred_packed = predict_packed(x, w1_demo, w2_demo, b_demo)
# 算损失：真实值减去预测值，取绝对值
loss_packed = torch.abs(y_demo - y_pred_packed)

# 循环打印各个变量的形状和值，方便对比看两种方式是不是一样
for name, value in [('x1_demo', x1_demo), ('x2_demo', x2_demo),
                    ('x', x), ('x[0]', x[0]), ('x[1]', x[1]),
                    ('y', y_demo), ('y_pred_packed', y_pred_packed),
                    ('loss_packed', loss_packed)]:
    # 打印变量名、形状、具体数值
    print(name, 'shape =', tuple(value.shape), 'value =', value)

# 对比两种写法的预测结果
print('两个输入张量的预测：', y_pred_demo)      # 分开写两个输入的结果
print('一个打包张量的预测：', y_pred_packed)     # 打包成一个张量的结果

# 断言检查（如果不成立会直接报错，成立就说明两种写法完全等价）
assert x.shape == (2,)                          # 检查 x 是不是一维、2个元素
assert torch.allclose(y_pred_demo, y_pred_packed)  # 两种方式预测值相等
assert torch.allclose(loss_demo, loss_packed)     # 两种方式损失值相等


x1_demo shape = () value = tensor(1.)
x2_demo shape = () value = tensor(2.)
x shape = (2,) value = tensor([1., 2.])
x[0] shape = () value = tensor(1.)
x[1] shape = () value = tensor(2.)
y shape = () value = tensor(-3.)
y_pred_packed shape = () value = tensor(3.)
loss_packed shape = () value = tensor(6.)
两个输入张量的预测： tensor(3.)
一个打包张量的预测： tensor(3.)


## 改变一个输入值，比较两种写法

先预测：只把第二个输入加 1，两种写法的预测是否仍然相同？第一个输入是否改变？

下面始终使用同一组参数，不进行训练。用 clone 复制 x 后再修改，以保留原始样本，方便比较。


In [4]:
# clone() 复制一份 x，得到 x_changed；这样不会改动原来的 x，方便前后对比
x_changed = x.clone()

# 把 x_changed 的第 2 个元素（下标从 0 开始，所以是 [1]，对应原来的 x2）加 1
# 相当于只修改第二个输入值，第一个输入（x[0]）保持不变
x_changed[1] = x_changed[1] + 1.0

# ===== 写法一：分开传两个输入 =====
# x1_demo 不变，x2_demo 加 1.0，手动把两个参数传进去
y_pred_separate_changed = predict(
    x1_demo, x2_demo + 1.0, w1_demo, w2_demo, b_demo)

# ===== 写法二：打包成一个张量传进去 =====
# 直接传修改后的 x_changed，函数内部用 x[0]、x[1] 取值
y_pred_packed_changed = predict_packed(
    x_changed, w1_demo, w2_demo, b_demo)

# ===== 打印对比 =====
print('原始 x: ', x, 'shape: ', tuple(x.shape))          # 原始 x 没被修改
print('修改后 x: ', x_changed, 'shape: ', tuple(x_changed.shape))  # 只有第二个元素变了
print('修改前的预测：', y_pred_packed)                       # 原始样本的预测值
print('修改后，两个输入张量的预测：', y_pred_separate_changed)  # 分开写法的新预测
print('修改后，一个打包张量的预测：', y_pred_packed_changed)     # 打包写法的新预测

# 断言：两种写法算出来的预测值完全相等
# 如果不相等会报错，相等就说明两种写法完全等价
assert torch.allclose(y_pred_separate_changed, y_pred_packed_changed)


原始 x:  tensor([1., 2.]) shape:  (2,)
修改后 x:  tensor([1., 3.]) shape:  (2,)
修改前的预测： tensor(3.)
修改后，两个输入张量的预测： tensor(4.)
修改后，一个打包张量的预测： tensor(4.)


## 我的观察

1. 两个标量张量的 shape 分别是什么？打包后的 x 的 shape 是什么？x[0] 和 x[1] 的 shape 又是什么？（20分）
2. x 的 shape 为 `(2,)`，在本例中代表两个样本，还是一个样本的两个输入值？（10分）
3. 比较 predict 与 predict_packed：数据的组织和传入方式如何改变？预测公式、参数、y 和 loss 是否改变？（20分）

待填写。


In [ ]:


- `x1_demo`、`x2_demo` 两个标量张量的 shape 都是0 维，就是一个单独的数
- 打包后的 `x` 的 shape 是一维，包含 2 个元素
- `x[0]` 和 `x[1]` 的 shape 又变回从张量里取出来之后，又变成 0 维标量了

2. `x` 的 shape 为 `(2,)` 代表什么：

- 代表**一个样本的两个输入值**，不是两个样本。
- 因为题目里同一个样本有两个输入 x1、x2，打包后就是一个长度为 2 的向量。如果是两个样本，每个样本两个输入，shape 应该是 `(2, 2)` 才对。

**3. `predict` 与 `predict_packed` 的对比：**

- **数据组织方式**：`predict` 把 x1、x2 分成两个独立的参数传进去；`predict_packed` 把两个输入打包成一个张量 `x`，在函数内部用 `x[0]`、`x[1]` 取值。
- **预测公式**：完全一样，都是 `w1*x1 + w2*x2 + b`。
- **参数**：`w1`、`w2`、`b` 都没有任何改变。
- **y 和 loss**：数值完全相同（`assert` 也验证了两种写法结果一致）。区别只是数据怎么 "打包传进去"，最终算出来的结果一模一样。